# Vídeo 3 – Agentes de IA e Grafos
## Parte 1 – Agente simples

In [5]:
"""
Vídeo 3 – Parte 1: Agente simples (conceito)
Foco: mostrar o ciclo Raciocínio ↔ Ação e o fallback sem LLM.
"""

import os
from typing import Dict
from dotenv import load_dotenv

# Pontos-chave: carregar .env e permitir rodar sem LLM (fallback).
try:
    from langchain_openai import ChatOpenAI
    from langchain.prompts import PromptTemplate
except Exception:
    ChatOpenAI = None
    PromptTemplate = None

load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2, max_tokens=128) if ChatOpenAI else None

# AÇÃO (ferramenta externa simulada): representa "chamar API".
def chamar_api_clima(cidade: str) -> Dict:
    return {"cidade": cidade, "previsao": "Ensolarado", "temp": 27}

def agente_conceito(mensagem: str) -> str:
    # RACIOCÍNIO (roteamento simples por intenção)
    msg = mensagem.lower()
    acao = "chamar_api_clima" if ("clima" in msg or "tempo" in msg) else "responder_direto"

    # AÇÃO: executar a ferramenta ou responder direto
    if acao == "chamar_api_clima":
        dados = chamar_api_clima("São Paulo")

        # Quando há LLM, usar PromptTemplate | LLM para verbalizar a resposta
        if llm and PromptTemplate:
            prompt = PromptTemplate.from_template("Explique em uma frase clara a previsão: {dados}")
            return (prompt | llm).invoke({"dados": str(dados)}).content

        # Fallback sem LLM
        return f"Previsão para {dados['cidade']}: {dados['previsao']}, {dados['temp']}°C."

    else:
        if llm and PromptTemplate:
            prompt = PromptTemplate.from_template("Responda de forma útil à pergunta: {mensagem}")
            return (prompt | llm).invoke({"mensagem": mensagem}).content

        # Fallback sem LLM
        return "Posso ajudar com clima, faturas ou cancelamentos."

if __name__ == "__main__":
    # Dois exemplos: um ativa ferramenta (clima) e outro segue resposta direta
    for e in ["Qual a previsão do tempo?", "Oi, tudo bem?"]:
        print("Usuário:", e)
        print("Agente:", agente_conceito(e))
        print("-" * 50)


Usuário: Qual a previsão do tempo?
Agente: A previsão do tempo para São Paulo indica um dia ensolarado com temperatura de 27 graus Celsius.
--------------------------------------------------
Usuário: Oi, tudo bem?
Agente: Oi! Tudo bem, obrigado por perguntar. E você, como está? Se precisar de algo ou tiver alguma pergunta, estou aqui para ajudar!
--------------------------------------------------


## Parte 2 – Sistema único em grafo

In [6]:
"""
Vídeo 3 – Parte 2: Um sistema único em grafo (LangGraph)
Etapas/nós + arestas condicionais + estado compartilhado.
"""

import os
from typing import Dict
from typing_extensions import TypedDict, Annotated
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Ponto-chave: permitir rodar com/sem LLM (demonstração não depende dele)
try:
    from langchain_openai import ChatOpenAI
    from langchain.prompts import PromptTemplate
except Exception:
    ChatOpenAI = None
    PromptTemplate = None

load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2, max_tokens=128) if ChatOpenAI else None


# Ponto-chave: "contrato" de estado compartilhado.
# Tudo que um nó produz/consome passa por aqui.
class State(TypedDict, total=False):
    mensagem: str          # entrada do usuário
    cliente: dict          # ex.: {"id": "123", "autenticado": True, "vip": False}
    intencao: str          # "faq" | "fatura" | "cancelar"
    fatura: dict           # dados vindos da API
    resposta: str          # texto final a mostrar ao usuário
    oferta: str            # proposta de retenção (se VIP)
    historico: Annotated[list, add_messages]  # opcional para demos com mensagens


# Ponto-chave: "ações" externas simuladas (APIs)
def api_consultar_fatura(cliente_id: str) -> Dict:
    return {"valor": 120.50, "vencimento": "10/09/2025"}
def api_registrar_cancelamento(cliente_id: str) -> bool: return True
def api_enviar_oferta(cliente_id: str, oferta: str) -> bool: return True


# --------------------------- NÓS (cada um faz 1 coisa) ---------------------------

def classificar_intencao(s: State) -> State:
    # Simplificação: regra por palavra-chave (no real, um LLM faria isso).
    msg = (s.get("mensagem") or "").lower()
    if "fatura" in msg: intent = "fatura"
    elif "cancel" in msg: intent = "cancelar"
    else: intent = "faq"
    return {"intencao": intent}

def responder_faq(s: State) -> State:
    # Nó terminal simples: produz "resposta".
    return {"resposta": "Resposta curta à FAQ."}

def consultar_fatura(s: State) -> State:
    # Guarda de segurança: se não autenticado, termina cedo com mensagem.
    if not s.get("cliente", {}).get("autenticado"):
        return {"resposta": "Faça login para ver a fatura."}
    # Caso autenticado, chama API e grava no estado.
    return {"fatura": api_consultar_fatura(s["cliente"]["id"])}

def resumir_fatura(s: State) -> State:
    # Converte dados brutos -> texto amigável (aqui simplificado).
    f = s.get("fatura", {})
    return {"resposta": f"Resumo da fatura: valor {f.get('valor')} venc. {f.get('vencimento')}"}

def decidir_vip(s: State) -> State:
    # Nó puramente de roteamento (não altera estado).
    return {}

def gerar_oferta(s: State) -> State:
    # Exemplo de uso combinado LLM+API (aqui mock).
    return {"oferta": "Desconto de 20% por 3 meses", "resposta": "Oferta enviada."}

def enviar_oferta(s: State) -> State:
    # Poderia chamar api_enviar_oferta(...). Mantemos simples.
    return {}

def registrar_cancelamento(s: State) -> State:
    # Terminal: registra e devolve resposta final.
    return {"resposta": "Cancelamento registrado."}


# --------------------------- GRAFO (arestas e fluxo) ---------------------------

g = StateGraph(State)

# Ponto-chave: cada função acima vira um nó nomeado.
g.add_node("classificar", classificar_intencao)
g.add_node("responder_faq", responder_faq)
g.add_node("consultar_fatura", consultar_fatura)
g.add_node("resumir_fatura", resumir_fatura)
g.add_node("decidir_vip", decidir_vip)
g.add_node("gerar_oferta", gerar_oferta)
g.add_node("enviar_oferta", enviar_oferta)
g.add_node("registrar_cancelamento", registrar_cancelamento)

# Ponto-chave: ponto de entrada único do fluxo.
g.set_entry_point("classificar")

# Ponto-chave: arestas condicionais = if/else declarativo (não espalhado no código).
def rota_intencao(s: State): return s.get("intencao", "faq")
g.add_conditional_edges(
    "classificar",
    rota_intencao,
    {"faq": "responder_faq", "fatura": "consultar_fatura", "cancelar": "decidir_vip"},
)

# Encadeamento linear entre nós quando não há condição.
g.add_edge("consultar_fatura", "resumir_fatura")

# Mais um roteamento condicional (VIP vs normal).
def rota_vip(s: State): return "vip" if s.get("cliente", {}).get("vip") else "normal"
g.add_conditional_edges(
    "decidir_vip",
    rota_vip,
    {"vip": "gerar_oferta", "normal": "registrar_cancelamento"},
)

# Continuação natural do fluxo após gerar oferta.
g.add_edge("gerar_oferta", "enviar_oferta")

# Ponto-chave: declarar nós terminais (chegada em END).
for end in ["responder_faq", "resumir_fatura", "enviar_oferta", "registrar_cancelamento"]:
    g.add_edge(end, END)

# Compila o grafo para execução.
app = g.compile()


if __name__ == "__main__":
    # Visualização do fluxo (ASCII) ajuda a explicar o grafo na aula.
    print(app.get_graph().draw_ascii())

    # Execução de exemplo (caminho "fatura")
    exemplo = {
        "mensagem": "Quero ver a fatura",
        "cliente": {"id": "123", "autenticado": True, "vip": False},
    }
    print(app.invoke(exemplo))


                                                  +-----------+                                                  
                                                  | __start__ |                                                  
                                                  +-----------+                                                  
                                                        *                                                        
                                                        *                                                        
                                                        *                                                        
                                                +-------------+                                                  
                                                | classificar |.....                                             
                                            ....+-------------+.    ........            